# Cheat Sheet: Stock Screening & Relative Valuation (Python/pandas)

Quick-reference snippets for this lab's tasks, with the R equivalent from the
original book noted where relevant. Not meant to be run top-to-bottom —
each cell is a standalone recipe.


## Imports & setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)


## Reading data

| Task | Python (pandas) | R (book) |
|---|---|---|
| Read local CSV | `pd.read_csv("finviz.csv")` | `read.csv("path/finviz.csv")` |
| Read CSV from a URL | `pd.read_csv("https://.../export.ashx?...")` | `read.csv(url(url_to_open))` |


In [2]:
# finviz = pd.read_csv("finviz.csv")
# finviz = pd.read_csv("https://elite.finviz.com/export.ashx?v=152&c=0,1,...,68&auth=KEY")


## First look / summary statistics

In [3]:
# df.head(n), df.tail(n)
# df.info()                      # dtypes + non-null counts  ~ R's str()
# df.describe(include="all").T   # numeric + categorical summary ~ R's summary()
# df.shape, df.columns.tolist()


## Cleaning messy numeric strings

R: `gsub("%|\\$|,|\\)|\\(", "", s)` then `as.numeric(s)`


In [4]:
def clean_numeric(series: pd.Series) -> pd.Series:
    cleaned = (series.astype(str)
                      .str.replace("%", "", regex=False)
                      .str.replace("$", "", regex=False)
                      .str.replace(",", "", regex=False)
                      .str.strip()
                      .replace("-", np.nan))
    return pd.to_numeric(cleaned, errors="coerce")

# Apply to many columns at once (R: apply(df[,7:68], 2, clean_numeric))
# for col in numeric_cols:
#     df[col] = clean_numeric(df[col])


In [5]:
def clean_market_cap(series: pd.Series) -> pd.Series:
    def _conv(x):
        x = str(x).strip()
        if x.endswith("B"): return float(x[:-1]) * 1000
        if x.endswith("M"): return float(x[:-1])
        try: return float(x)
        except ValueError: return np.nan
    return series.apply(_conv)


## Group aggregation (R's `aggregate()` / `ddply()`)

| Task | pandas | R |
|---|---|---|
| Mean of one column by group | `df.groupby("Sector")["Price"].mean()` | `aggregate(Price~Sector, data=df, FUN="mean")` |
| Mean of several columns by group | `df.groupby("Sector")[cols].mean()` | `ddply(df, "Sector", summarise, ...)` |
| Group by two keys | `df.groupby(["Sector","Industry"])[cols].mean()` | `aggregate(Price~Sector+Industry, ...)` |


In [6]:
# sector_avg = df.groupby("Sector")[metrics].mean().add_prefix("SAvg_").reset_index()
# industry_avg = df.groupby(["Sector","Industry"])[metrics].mean().add_prefix("IAvg_").reset_index()


## Reshaping: long ↔ wide (R's `melt()` / `dcast()`)

| Task | pandas | R |
|---|---|---|
| Wide → long | `df.melt(id_vars=["Sector"], var_name="variable", value_name="value")` | `melt(df, id="Sector")` |
| Long → wide (aggregating) | `df.pivot_table(index="Sector", columns="variable", values="value", aggfunc="mean")` | `dcast(df, Sector~variable, mean)` |


In [7]:
# long_df = df.melt(id_vars=["Sector"], var_name="variable", value_name="value")
# wide_df = long_df.pivot_table(index="Sector", columns="variable", values="value", aggfunc="mean").reset_index()


## Merging benchmark columns back onto the main dataframe

In [8]:
# df = df.merge(sector_avg, on="Sector").merge(industry_avg, on=["Sector","Industry"])


## Flagging / composite scoring

In [9]:
# df["S_PE_Under"] = (df["P/E"] < df["SAvg_P/E"]).astype(int)
# flag_cols = [c for c in df.columns if c.endswith("_Under")]
# df["RelValIndex"] = df[flag_cols].sum(axis=1)


## Multi-criteria filtering (screening)

In [10]:
# target = df[
#     (df.Price.between(20, 100)) &
#     (df.Volume > 10_000) &
#     (df.Country == "USA") &
#     (df["Total Debt/Equity"] < 1) &
#     (df.Beta < 1.5) &
#     (df.RelValIndex >= 8)
# ]


## Rolling / moving averages (R's `zoo::rollmean`)

| Task | pandas | R |
|---|---|---|
| N-period moving average | `s.rolling(N).mean()` | `rollmean(s, N, align="right")` |
| Per-group moving average | `df.groupby("Symbol")["Close"].transform(lambda s: s.rolling(N).mean())` | loop + `rollmean` per symbol |


In [11]:
# df["MA50"] = df.groupby("Symbol")["AdjClose"].transform(lambda s: s.rolling(50).mean())
# df["MA200"] = df.groupby("Symbol")["AdjClose"].transform(lambda s: s.rolling(200).mean())


## Plotting (matplotlib ~ ggplot2)

| Task | matplotlib | R/ggplot2 |
|---|---|---|
| Histogram | `plt.hist(s, bins=100)` | `hist(x, breaks=100)` |
| Bar chart | `plt.bar(x, y)` | `geom_bar(stat="identity")` |
| Line chart, multi-series | loop `plt.plot(x, y, label=...)` + `plt.legend()` | `geom_line(aes(color=variable))` |
| Rotate x labels | `plt.xticks(rotation=90)` | `theme(axis.text.x = element_text(angle = 90))` |


In [12]:
# fig, ax = plt.subplots()
# ax.hist(df["Price"][df["Price"] < 150], bins=100)
# ax.set_title("Price Distribution"); ax.set_xlabel("Price"); ax.set_ylabel("Frequency")
# plt.show()


In [13]:
# fig, ax = plt.subplots()
# for sym in tickers:
#     sub = df[df.Symbol == sym]
#     ax.plot(sub.Date, sub.AdjClose, label=sym)
# ax.legend(); plt.show()


## Outlier handling patterns

In [14]:
# Find the row with the max value of a column
# outlier_row = df.loc[df["Price"].idxmax()]

# Drop it
# df = df[df.Ticker != outlier_row.Ticker].reset_index(drop=True)

# Alternative: winsorize/cap instead of dropping
# df["Price_capped"] = df["Price"].clip(upper=df["Price"].quantile(0.99))


## Common gotchas

- `pd.read_csv` won't auto-clean `%`/`$`/`,` — those columns load as `object`
  (string) dtype; always check `.dtypes` after loading.
- `groupby(...).mean()` silently drops non-numeric columns unless you select
  columns first (`df.groupby("Sector")[metrics].mean()`).
- `merge()` defaults to an **inner join** — rows without a match on the join
  keys (e.g. no industry average available) are silently dropped. Check
  `len(df)` before/after merges.
- `.rolling(N).mean()` needs at least N rows per group before it produces a
  value — expect `NaN` for the first N-1 rows of each symbol.
- Real finviz market cap strings use suffixes like `B`/`M` that `clean_numeric`
  alone won't parse — always handle Market Cap with a dedicated converter.
